In [ ]:
# Some derivations for the braid matrix

# ((tt -> a) at -> c) ((tt -> b) bt -> d) cd -> 1
()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh

phi = (1 + np.sqrt(5)) / 2
exact_E_per_site = np.sqrt(5) - 3

def get_basis(N):
    basis = []
    for i in range(2**N):
        s = bin(i)[2:].zfill(N)
        if '00' not in s:
            if s[0] == '0' and s[-1] == '0':
                continue
            basis.append(s)
    return basis

def build_hamiltonian(N, basis):
    dim = len(basis)
    H = lil_matrix((dim, dim), dtype=float)
    basis_dict = {s: i for i, s in enumerate(basis)}
    
    for i in range(N):
        prev_i = (i - 1) % N
        next_i = (i + 1) % N
        
        for j, s in enumerate(basis):
            if s[prev_i] == '1' and s[next_i] == '1':
                if s[i] == '0':
                    H[j, j] -= 1.0 / (phi**2)
                    s_new = s[:i] + '1' + s[i+1:]
                    k = basis_dict[s_new]
                    H[k, j] -= 1.0 / (phi**1.5)
                elif s[i] == '1':
                    H[j, j] -= 1.0 / phi
                    s_new = s[:i] + '0' + s[i+1:]
                    k = basis_dict[s_new]
                    H[k, j] -= 1.0 / (phi**1.5)
            elif s[prev_i] == '0' and s[next_i] == '0':
                H[j, j] -= 1.0
                
    return H.tocsr()

Ns = range(3, 51, 10)
Es = []

for N in Ns:
    basis = get_basis(N)
    H = build_hamiltonian(N, basis)
    
    if H.shape[0] <= 2:
        w, v = np.linalg.eigh(H.toarray())
        e0 = w[0]
    else:
        w, v = eigsh(H, k=1, which='SA')
        e0 = w[0]
    
    Es.append(e0 / N)


#plt.plot(even_df.N, even_df.E, 'bo-', label='Even N')
plt.plot(Ns, Es, 'go-', label='Odd N')
plt.axhline(exact_E_per_site, color='r', linestyle='--', label=f'Exact limit $\\sqrt{{5}}-3 \\approx {exact_E_per_site:.4f}$')
plt.xlabel('Chain Length N')
plt.ylabel('Ground State Energy per Site ($E_0/N$)')
plt.title('Fibonacci Golden Chain')
plt.legend()
plt.grid(True)
plt.savefig('convergence_plot.png')